# Official Anthropic J-Lens × Qwen validation

This validates Anthropic's official library and published lens before any from-scratch work. Use a Colab L4 or A100 runtime, or the validated T4 fallback.

## 1 · Reproducible environment

In [ ]:
%pip install -q "git+https://github.com/anthropics/jacobian-lens.git@581d398613e5602a5af361e1c34d3a92ea82ba8e" ipython

In [ ]:
import json
from pathlib import Path
import torch, transformers, jlens
from huggingface_hub import hf_hub_download
assert torch.cuda.is_available(), "Select Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
Path("artifacts").mkdir(exist_ok=True)
jlens.configure_logging()

## 2 · Matched model and published lens

This lens was fitted on 1,000 WikiText sequences and must be paired with its exact model.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
MODEL_REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_FILE = "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt"
LENS_SOURCE_REF = "qwen-n1000"  # human-readable provenance only
LENS_REVISION = "16a01f309fcec900fdcec3f4cd5b64f3d00e4d5a"
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, dtype=torch.bfloat16, device_map="cuda"
).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
)
resolved_config_path = Path(hf_hub_download(
    MODEL_NAME, "config.json", revision=MODEL_REVISION
))
snapshot_index = resolved_config_path.parts.index("snapshots")
RESOLVED_MODEL_REVISION = resolved_config_path.parts[snapshot_index + 1]
assert RESOLVED_MODEL_REVISION == MODEL_REVISION, (RESOLVED_MODEL_REVISION, MODEL_REVISION)
model = jlens.from_hf(hf_model, tokenizer)
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
print("model revision:", RESOLVED_MODEL_REVISION)
print("lens revision:", LENS_REVISION, f"(source ref: {LENS_SOURCE_REF})")
print(model, lens)

## 3 · Next-token position, parity gate, and controlled comparison

Causal-LM logits at sequence position `p` predict token `p + 1`. These prompts end immediately before the completion we want to inspect, so the correct residual/logit position is the **last supplied token**, `-1`. Position `-2` would predict the final token already present in the prompt and would turn the evaluation into teacher-forced reconstruction.

The direct model and official API must agree at `-1`. J-Lens and logit lens then differ only by application of the fitted Jacobian.


In [ ]:
PROMPTS = {
 "multihop": "Fact: The currency used in the country shaped like a boot is",
 "multilingual": "Translate into French: The small red house is",
 "typo": "Correct the typo and complete: The capital of Frnace is",
 "ordering": "Put in increasing order: 9, 2, 5. The first number is",
 "ascii_face": "This is a face:\n  ^  ^\n   ^\n  ___\nThe middle ^ is the",
}
layers = [model.n_layers // 4, model.n_layers // 2, 3 * model.n_layers // 4, model.n_layers - 2]
READOUT_POSITION = -1  # final supplied token; its logits predict the first unseen token
def top_tokens(logits, k=10):
 ids = logits.float().topk(k).indices.tolist()
 return [tokenizer.decode([i], clean_up_tokenization_spaces=False) for i in ids]
report = {
 "model": MODEL_NAME,
 "model_revision": MODEL_REVISION,
 "resolved_model_revision": RESOLVED_MODEL_REVISION,
 "lens_revision": LENS_REVISION,
 "lens_source_ref": LENS_SOURCE_REF,
 "readout_position": READOUT_POSITION,
 "readout_semantics": "logits after the final supplied token predict the first unseen token",
 "prompts": {},
}
for name, prompt in PROMPTS.items():
 j_logits, api_logits, _ = lens.apply(model, prompt, layers=layers, positions=[READOUT_POSITION])
 ll_logits, _, _ = lens.apply(model, prompt, layers=layers, positions=[READOUT_POSITION], use_jacobian=False)
 ids = model.encode(prompt, max_length=512)
 with torch.no_grad(): direct = hf_model(input_ids=ids).logits[0, READOUT_POSITION].float().cpu()
 final = api_logits[0].float().cpu()
 error = (direct - final).abs().max().item()
 assert torch.allclose(direct, final, atol=0.25, rtol=0.01), error
 rows = {}
 print(f"\n{name}: {prompt!r}")
 print("readout token:", tokenizer.decode([ids[0, READOUT_POSITION].item()]), "→ predicts next unseen token")
 for layer in layers:
  j, ll = j_logits[layer][0].float().cpu(), ll_logits[layer][0].float().cpu()
  overlap = len(set(j.topk(10).indices.tolist()) & set(ll.topk(10).indices.tolist())) / 10
  rows[str(layer)] = {"jlens_top10": top_tokens(j), "logit_lens_top10": top_tokens(ll), "top10_overlap": overlap}
  print(f"L{layer:>2} J:", top_tokens(j, 5)); print(f"L{layer:>2} L:", top_tokens(ll, 5))
 report["prompts"][name] = {"model_parity_max_abs_error": error, "model_top10": top_tokens(final), "layers": rows}
Path("artifacts/qwen_validation.json").write_text(json.dumps(report, indent=2))

## 4 · Paper-style interactive slice

Cell labels are top tokens and ranks remain full-vocabulary ranks. Click cells to pin tokens and inspect trajectories.

In [ ]:
from jlens.vis import build_page, compute_slice, notebook_iframe
prompt = PROMPTS["ascii_face"]
slice_data = compute_slice(model, lens, prompt, layer_stride=2, top_n=10, mask_display=True)
page, _, _ = build_page(
 slice_data, prompt, title="Qwen official J-Lens · ASCII-face validation",
 description="Official n=1,000 lens. Click a cell to inspect rank across layer and position.", mode="embed"
)
Path("artifacts/qwen_ascii_face_slice.html").write_text(page)
notebook_iframe(page, height=720)

## 5 · Interpretation

A run passes when parity passes, every prompt family completes, both artifacts save, and the interactive page renders. Semantic readouts are descriptive, not causal evidence.